**相比 V2.1 的变化**：放弃 PDF→Markdown 转换，因为提取的文字不含准确的标题信息，太容易把不是标题的当成标题了，所以改为直接提取纯文本并清洗，使用 `chunk_text` 切分。

**验证函数**：`ingestion/extractors/pdf.py` 的 `extract_text()`

**验证样本**：
- 样本 A：英文双栏学术论文（*Learning Agile Robotic Locomotion Skills by Imitating Animals*, 2020）
- 样本 B：中文规则文档（2026 ROBOCON 仿生足式机器人挑战赛规则 V2.0）

**关注点**：
1. `(cid:xx)` 公式占位符是否被过滤
2. arXiv 水印噪声行（`l u J` 等）是否被过滤
3. 单词/字符碎片是否减少
4. 正常正文段落是否保留完整


In [1]:
import sys
sys.path.insert(0, '..')

from ingestion.extractors.pdf import extract_text, make_doc_id
print('导入成功')

导入成功


In [2]:
PDF_A = '../experience_data/2020 - Learning Agile Robotic Locomotion Skills by Imitating Animals.pdf'

text_a = extract_text(PDF_A)
lines_a = text_a.splitlines()
print(f'总字符数: {len(text_a)}')
print(f'总行数  : {len(lines_a)}')
print(f'段落数  : {text_a.count(chr(10)*2) + 1}')

总字符数: 57109
总行数  : 267
段落数  : 134


In [3]:
# 查看前 60 行
for i, line in enumerate(lines_a[:60]):
    print(f'{i+1:3d} | {line}')

  1 | Learning Agile Robotic Locomotion Skills by Imitating Animals Xue Bin Peng∗†, Erwin Coumans∗, Tingnan Zhang∗, Tsang-Wei Edward Lee∗, Jie Tan∗, Sergey Levine∗† †University of California, Berkeley Email: xbpeng@berkeley.edu, {erwincoumans,tingnan,tsangwei,jietan}@google.com, svlevine@eecs.berkeley.edu Fig. 1. Laikago robot performing locomotion skills learned by imitating motion data recorded from a real dog. Top: Motion capture data recorded from a dog. Middle: Simulated Laikago robot imitating reference motions. Bottom: Real Laikago robot imitating reference motions.
  2 | 
  3 | Abstract—Reproducing the diverse and agile locomotion skills of animals has been a longstanding challenge in robotics. While manually-designed controllers have been able to emulate many complex behaviors, building such controllers involves a timeconsuming and difﬁcult development process, often requiring substantial expertise of the nuances of each skill. Reinforcement learning provides an appealing alte

In [4]:
PDF_B = '../experience_data/2026年第二十五届全国大学生机器人大赛ROBOCON仿生足式机器人挑战赛比赛规则-V2.0.pdf'

text_b = extract_text(PDF_B)
lines_b = text_b.splitlines()
print(f'总字符数: {len(text_b)}')
print(f'总行数  : {len(lines_b)}')
print(f'段落数  : {text_b.count(chr(10)*2) + 1}')

总字符数: 11129
总行数  : 291
段落数  : 146


In [5]:
# 查看前 60 行
for i, line in enumerate(lines_b[:60]):
    print(f'{i+1:3d} | {line}')

  1 | 第二十五届全国大学生机器人大赛ROBOCON 仿生足式机器人挑战赛比赛规则全国大学生机器人大赛组委会2025 年 11 月修订日期说明修订历史3/11/2025 在全国大学生机器人大赛 ROBOCON 官网发布：www.robocon.org.cn 25/11/2025 本次给出的修改内容仅限于规则的具体调整，至于文字措辞的变化不再说明。具体规则修正如下：
  2 | 
  3 | 3.1 障碍赛场地中，“直角绕杆”的必达区修改为红色虚线圆，必达区内
  4 | 
  5 | 部颜色与障碍赛场地同色；“大斜坡”的角度修改为 11.3；
  6 | 
  7 | 3.2 任务赛场地中，图 3、图 4 中间隔离墙的颜色改为橘黄色；减速带颜
  8 | 
  9 | 色修改为黄黑相间；物资箱的重量为 0.5Kg。
 10 | 
 11 | 4. “机器人”第(7)条，机器人的外形尺寸修改为“机器人在自然站立状态下的外形尺寸不得超过 800mm600mm600mm”。
 12 | 
 13 | 6.1 赛前准备第(5)条，修改为“机器人自主识别智力题目的时间不得超过
 14 | 
 15 | 20 秒。该时间不计算在赛前准备的 1 分钟内”。
 16 | 
 17 | 6.3 障碍赛表 2 中，对大斜坡越障成功的定义修改为“机器人在两侧斜坡上
 18 | 
 19 | 沿 3m 长边方向分别行走的距离不得少于 1 米”。
 20 | 
 21 | 6.4 任务赛(6)条，修改为“机器人需要通过机器人的视觉识别智力题目并通
 22 | 
 23 | 过显示屏或声音播放给出计算答案”。
 24 | 
 25 | 8.取消比赛资格第(2)条，修改为“一个学校若有多个参赛队，各参赛队之间的机器人本体结构明显雷同者”。
 26 | 
 27 | 删除了 11.附图。
 28 | 
 29 | 目录1. 赛事简介 ........................................................................................................................................................... 4 2. 赛项设置 ..............